In [ ]:
%matplotlib widget

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider, HTML, HTMLMath,
    VBox, HBox, Layout
)
from IPython.display import display

plt.ioff()

# ============================================================
# DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.gs-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.gs-label {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.gs-value {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial,sans-serif;
    font-size:15px;
    line-height:1.5;
    margin-bottom:10px;
">

<div class="gs-title" style="margin-bottom:8px;">
Gram–Schmidt Orthogonalization
</div>

<div style="margin-bottom:5px;">
Starting from two linearly independent vectors u₁ and u₂,
the Gram–Schmidt procedure constructs an orthogonal pair
w₁,w₂ spanning exactly the same subspace.
</div>

<div style="margin-bottom:5px;">
The second vector is obtained by removing from u₂ its
projection onto w₁. The resulting vectors are then normalized
to obtain the orthonormal basis e₁,e₂.
</div>

<div>
<b>This notebook:</b> computes the projection, orthogonalization,
normalization and orthogonality checks symbolically using SymPy.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

def make_slider(value):
    return IntSlider(
        min=-5,
        max=5,
        step=1,
        value=value,
        readout=False,
        continuous_update=True,
        layout=Layout(width='300px')
    )

u11 = make_slider(-3)
u12 = make_slider(2)

u21 = make_slider(-2)
u22 = make_slider(1)

u11_value = HTML()
u12_value = HTML()
u21_value = HTML()
u22_value = HTML()

# ============================================================
# CONTROL ROW
# ============================================================

def control_row(label, slider, value_widget):

    return HBox(
        [
            HTML(
                f'<div class="gs-label">{label}</div>',
                layout=Layout(
                    width='55px',
                    min_width='55px'
                )
            ),

            slider,

            value_widget
        ],
        layout=Layout(
            width='520px',
            height='38px',
            align_items='center'
        )
    )

# ============================================================
# INPUT PANEL
# ============================================================

vector1_controls = VBox(
    [
        HTML(
            '<div class="gs-title" '
            'style="font-size:17px;margin-bottom:3px;">'
            'Vector u₁'
            '</div>'
        ),

        control_row('u₁x:', u11, u11_value),
        control_row('u₁y:', u12, u12_value)
    ],
    layout=Layout(width='540px')
)

vector2_controls = VBox(
    [
        HTML(
            '<div class="gs-title" '
            'style="font-size:17px;margin-bottom:3px;">'
            'Vector u₂'
            '</div>'
        ),

        control_row('u₂x:', u21, u21_value),
        control_row('u₂y:', u22, u22_value)
    ],
    layout=Layout(width='540px')
)

controls_row = HBox(
    [
        vector1_controls,
        vector2_controls
    ],
    layout=Layout(
        width='1120px',
        gap='20px',
        align_items='flex-start'
    )
)

parameters_panel = VBox(
    [
        HTML(
            '<div class="gs-title" '
            'style="margin-bottom:6px;">'
            'Input Vectors'
            '</div>'
        ),

        controls_row
    ],
    layout=Layout(
        width='1140px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

# ============================================================
# LATEX HELPERS
# ============================================================

def scalar_latex(expr):

    result = sp.latex(
        sp.simplify(expr)
    )

    result = result.replace(
        r'\frac',
        r'\dfrac'
    )

    return result


def vector_latex(vector):

    entries = [
        scalar_latex(vector[i])
        for i in range(vector.rows)
    ]

    body = r'\\'.join(entries)

    return (
        r'\bigg['
        r'\begin{array}{r}'
        +
        body
        +
        r'\end{array}'
        r'\bigg]'
    )

# ============================================================
# SYMBOLIC RESULT WIDGETS
# ============================================================

u1_math   = HTMLMath()
u2_math   = HTMLMath()
w1_math   = HTMLMath()
proj_math = HTMLMath()
w2_math   = HTMLMath()
e1_math   = HTMLMath()
e2_math   = HTMLMath()

check1_math = HTMLMath()
check2_math = HTMLMath()

conclusion_widget = HTML()

# ============================================================
# FIRST SYMBOLIC ROW
# ============================================================

symbolic_first_row = HBox(
    [
        u1_math,
        u2_math,
        w1_math,
        proj_math,
        w2_math,
        e1_math,
        e2_math
    ],
    layout=Layout(
        width='1110px',
        justify_content='space-between',
        align_items='center'
    )
)

# ============================================================
# CHECKS ROW
# ============================================================

checks_row = HBox(
    [
        check1_math,
        check2_math
    ],
    layout=Layout(
        width='720px',
        justify_content='space-around',
        align_items='center',
        margin='8px 0px 2px 280px'
    )
)

# ============================================================
# SYMBOLIC PANEL
# ============================================================

symbolic_panel = VBox(
    [
        HTML(
            '<div class="gs-title" '
            'style="margin-bottom:12px;">'
            'Symbolic Results'
            '</div>'
        ),

        symbolic_first_row,

        HTML("""
        <div style="
            width:1090px;
            height:1px;
            background:#d8d8d8;
            margin-top:14px;
            margin-bottom:10px;
        "></div>
        """),

        checks_row,

        conclusion_widget
    ],
    layout=Layout(
        width='1140px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

# ============================================================
# SYMBOLIC COMPUTATION
# ============================================================

def gram_schmidt():

    u1 = sp.Matrix([
        u11.value,
        u12.value
    ])

    u2 = sp.Matrix([
        u21.value,
        u22.value
    ])

    if u1 == sp.zeros(2,1):
        return (
            u1, u2,
            None, None, None,
            None, None,
            None, None
        )

    w1 = u1

    denominator = sp.simplify(
        w1.dot(w1)
    )

    projection = sp.simplify(
        (
            u2.dot(w1)
            /
            denominator
        )
        *
        w1
    )

    w2 = sp.simplify(
        u2
        -
        projection
    )

    if w2 == sp.zeros(2,1):
        return (
            u1, u2,
            w1, projection, w2,
            None, None,
            None, None
        )

    norm1 = sp.sqrt(
        sp.simplify(
            w1.dot(w1)
        )
    )

    norm2 = sp.sqrt(
        sp.simplify(
            w2.dot(w2)
        )
    )

    e1 = sp.simplify(
        w1 / norm1
    )

    e2 = sp.simplify(
        w2 / norm2
    )

    check_w = sp.simplify(
        w1.dot(w2)
    )

    check_e = sp.simplify(
        e1.dot(e2)
    )

    return (
        u1, u2,
        w1, projection, w2,
        e1, e2,
        check_w, check_e
    )

# ============================================================
# INITIAL DATA
# ============================================================

result = gram_schmidt()

u1_initial = result[0]
u2_initial = result[1]

u1n = np.array(
    u1_initial,
    dtype=float
).reshape(-1)

u2n = np.array(
    u2_initial,
    dtype=float
).reshape(-1)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.4, 5.4)
)

fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False

fig.canvas.layout = Layout(
    width='740px',
    height='540px',
    margin='0px'
)

ax.set_title(
    'Gram–Schmidt Geometry in R²',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax.set_xlabel('x₁')
ax.set_ylabel('x₂')

ax.set_xlim(-6,6)
ax.set_ylim(-6,6)

ax.set_aspect(
    'equal',
    adjustable='box'
)

ax.axhline(
    0,
    linewidth=0.8
)

ax.axvline(
    0,
    linewidth=0.8
)

ax.grid(
    True,
    linestyle=':',
    alpha=0.40
)

u1_line, = ax.plot(
    [0,u1n[0]],
    [0,u1n[1]],
    linewidth=2.2,
    marker='o',
    label='u₁'
)

u2_line, = ax.plot(
    [0,u2n[0]],
    [0,u2n[1]],
    linewidth=2.2,
    marker='o',
    label='u₂'
)

w1_line, = ax.plot(
    [],
    [],
    linewidth=2.4,
    marker='o',
    label='w₁'
)

w2_line, = ax.plot(
    [],
    [],
    linewidth=2.4,
    marker='o',
    label='w₂'
)

projection_line, = ax.plot(
    [],
    [],
    linewidth=1.7,
    linestyle=':',
    label='proj'
)

connection_line, = ax.plot(
    [],
    [],
    linewidth=1.2,
    linestyle=':'
)

ax.legend(
    loc='upper right'
)

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.12
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    width:1140px;
    padding:12px 15px;
    border:1px solid #d7c7e5;
    font-family:Arial,sans-serif;
    font-size:14px;
    line-height:1.60;
    box-sizing:border-box;
    margin-top:5px;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:8px;
">
Interpretation
</div>

<div style="margin-bottom:7px;">
The projection identifies the component of u₂ lying along w₁.
Subtracting that component leaves w₂, which is perpendicular to w₁.
</div>

<div style="margin-bottom:7px;">
The Gram–Schmidt process does not change the span of the vectors.
It replaces the original basis by an orthogonal basis spanning
exactly the same subspace.
</div>

<div>
Normalizing w₁ and w₂ produces the orthonormal basis e₁,e₂.
All projections, norms and orthogonality checks displayed above
are calculated symbolically by SymPy.
</div>

</div>
""")

# ============================================================
# UPDATE
# ============================================================

def update_notebook(change=None):

    u11_value.value = (
        f'<div class="gs-value">{u11.value}</div>'
    )

    u12_value.value = (
        f'<div class="gs-value">{u12.value}</div>'
    )

    u21_value.value = (
        f'<div class="gs-value">{u21.value}</div>'
    )

    u22_value.value = (
        f'<div class="gs-value">{u22.value}</div>'
    )

    (
        u1, u2,
        w1, projection, w2,
        e1, e2,
        check_w, check_e
    ) = gram_schmidt()

    u1n = np.array(
        u1,
        dtype=float
    ).reshape(-1)

    u2n = np.array(
        u2,
        dtype=float
    ).reshape(-1)

    u1_line.set_data(
        [0,u1n[0]],
        [0,u1n[1]]
    )

    u2_line.set_data(
        [0,u2n[0]],
        [0,u2n[1]]
    )

    if w1 is None:

        w1_line.set_data([],[])
        w2_line.set_data([],[])
        projection_line.set_data([],[])
        connection_line.set_data([],[])

        u1_math.value = (
            r'\(\mathbf{u}_1='
            +
            vector_latex(u1)
            +
            r'\)'
        )

        u2_math.value = (
            r'\(\mathbf{u}_2='
            +
            vector_latex(u2)
            +
            r'\)'
        )

        for widget in [
            w1_math, proj_math, w2_math,
            e1_math, e2_math,
            check1_math, check2_math
        ]:
            widget.value = ''

        conclusion_widget.value = """
        <div style="
            font-family:Arial;
            font-size:14px;
            line-height:1.5;
            margin-top:10px;
        ">
        <b>Conclusion:</b>
        Gram–Schmidt cannot start with the zero vector.
        </div>
        """

        fig.canvas.draw_idle()
        return

    w1n = np.array(
        w1,
        dtype=float
    ).reshape(-1)

    w1_line.set_data(
        [0,w1n[0]],
        [0,w1n[1]]
    )

    pn = np.array(
        projection,
        dtype=float
    ).reshape(-1)

    projection_line.set_data(
        [0,pn[0]],
        [0,pn[1]]
    )

    connection_line.set_data(
        [pn[0],u2n[0]],
        [pn[1],u2n[1]]
    )

    if w2 == sp.zeros(2,1):

        w2_line.set_data([],[])

        u1_math.value = (
            r'\(\mathbf{u}_1='
            + vector_latex(u1)
            + r'\)'
        )

        u2_math.value = (
            r'\(\mathbf{u}_2='
            + vector_latex(u2)
            + r'\)'
        )

        w1_math.value = (
            r'\(\mathbf{w}_1='
            + vector_latex(w1)
            + r'\)'
        )

        proj_math.value = (
            r'\(\mathrm{proj}_{\mathbf{w}_1}\mathbf{u}_2='
            + vector_latex(projection)
            + r'\)'
        )

        w2_math.value = (
            r'\(\mathbf{w}_2='
            + vector_latex(w2)
            + r'\)'
        )

        e1_math.value = ''
        e2_math.value = ''
        check1_math.value = ''
        check2_math.value = ''

        conclusion_widget.value = """
        <div style="
            font-family:Arial;
            font-size:14px;
            line-height:1.5;
            margin-top:10px;
        ">
        <b>Conclusion:</b>
        The input vectors are linearly dependent.
        The second Gram–Schmidt vector becomes zero.
        </div>
        """

        fig.canvas.draw_idle()
        return

    w2n = np.array(
        w2,
        dtype=float
    ).reshape(-1)

    w2_line.set_data(
        [0,w2n[0]],
        [0,w2n[1]]
    )

    u1_math.value = (
        r'\(\mathbf{u}_1='
        +
        vector_latex(u1)
        +
        r'\)'
    )

    u2_math.value = (
        r'\(\mathbf{u}_2='
        +
        vector_latex(u2)
        +
        r'\)'
    )

    w1_math.value = (
        r'\(\mathbf{w}_1='
        +
        vector_latex(w1)
        +
        r'\)'
    )

    proj_math.value = (
        r'\(\mathrm{proj}_{\mathbf{w}_1}\mathbf{u}_2='
        +
        vector_latex(projection)
        +
        r'\)'
    )

    w2_math.value = (
        r'\(\mathbf{w}_2='
        +
        vector_latex(w2)
        +
        r'\)'
    )

    e1_math.value = (
        r'\(\mathbf{e}_1='
        +
        vector_latex(e1)
        +
        r'\)'
    )

    e2_math.value = (
        r'\(\mathbf{e}_2='
        +
        vector_latex(e2)
        +
        r'\)'
    )

    check1_math.value = (
        r'\('
        r'\mathbf{w}_1^{T}\mathbf{w}_2='
        +
        scalar_latex(check_w)
        +
        r'\)'
    )

    check2_math.value = (
        r'\('
        r'\mathbf{e}_1^{T}\mathbf{e}_2='
        +
        scalar_latex(check_e)
        +
        r'\)'
    )

    conclusion_widget.value = """
    <div style="
        font-family:Arial;
        font-size:14px;
        line-height:1.5;
        margin-top:10px;
    ">
    <b>Conclusion:</b>
    The vectors w₁ and w₂ are orthogonal, while e₁ and e₂ form
    an orthonormal basis of the same subspace generated by u₁ and u₂.
    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

for slider in [
    u11, u12,
    u21, u22
]:
    slider.observe(
        update_notebook,
        names='value'
    )

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# FINAL LAYOUT
# ============================================================

figure_block = VBox(
    [
        fig.canvas
    ],
    layout=Layout(
        width='760px',
        margin='0px 0px 5px 0px'
    )
)

display(
    VBox(
        [
            documentation,
            parameters_panel,
            symbolic_panel,
            figure_block,
            interpretation
        ],
        layout=Layout(
            width='1180px',
            gap='8px',
            align_items='flex-start'
        )
    )
)